---
title: "Fishing Effort Within High Seas Marine Protected Area Candidates: A Focus on the South Atlantic"
subtitle: Reproducible analysis using cloud-native geospatial data
authors:
  - name: GeoAgent
    affiliations:
      - Boettiger Lab, UC Berkeley
  - name: Claude
    affiliations:
      - Anthropic, Inc.
keywords:
  - high seas
  - MPA
  - BBNJ
  - fishing effort
  - South Atlantic
  - Walvis Ridge
  - ABNJ
  - Global Fishing Watch
license: CC-BY-4.0
subject: Marine Conservation
abstract: |
  The 2023 UN High Seas Treaty (BBNJ Agreement) established a framework for
  creating marine protected areas (MPAs) in areas beyond national jurisdiction.
  We examine 18 proposed MPA candidates covering a combined 16.4 million km²,
  identify two candidates — the Atlantic Equatorial Fracture Zone and Walvis
  Ridge — within the South Atlantic, and quantify fishing effort inside the
  Walvis Ridge boundary from 2012 to 2024 using Global Fishing Watch AIS data.
  Fishing activity in the Walvis Ridge candidate roughly tripled between 2012
  and its 2021 peak of 52,728 hours before declining. These results highlight
  the importance of spatially explicit fishing-effort baselines for evaluating
  displacement risk and informing negotiations under the BBNJ process.
exports:
  - format: "typst"
    template: "lapreprint-typst"
    output: "fishing-south-atlantic.pdf"
---

In [1]:
import duckdb
import matplotlib.pyplot as plt
import pandas as pd

con = duckdb.connect()
con.execute("INSTALL h3 FROM community; LOAD h3;")
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET s3_endpoint='s3-west.nrp-nautilus.io';")
con.execute("SET s3_url_style='path';")
con.execute("SET s3_use_ssl=true;")

## Introduction

The high seas — ocean areas beyond national jurisdiction (ABNJ) — cover nearly half the Earth's surface yet have historically lacked formal conservation mechanisms. The 2023 Agreement under the United Nations Convention on the Law of the Sea on the Conservation and Sustainable Use of Marine Biological Diversity of Areas Beyond National Jurisdiction (BBNJ Agreement) [](doi:10.1163/15718085-bja10137) provides the first legal framework for establishing MPAs on the high seas.

Multiple candidate sites have been proposed through processes including Ecologically or Biologically Significant Areas (EBSAs) identification and regional championing by coastal states. Understanding existing human uses — particularly commercial fishing — within these candidate boundaries is essential for evaluating potential socioeconomic displacement and designing effective management measures.

This analysis addresses three questions:

1. What is the combined spatial footprint of all proposed high seas MPA candidates?
2. Which candidates fall within the South Atlantic Ocean?
3. What are the temporal trends in fishing effort within the Walvis Ridge candidate?

In [2]:
MPA_HEX = 's3://public-high-seas/mpa-candidates/hex/h0=*/data_0.parquet'

total_area = con.sql(f"""
    SELECT SUM(Area) AS total_area_km2
    FROM (
        SELECT DISTINCT _cng_fid, Area
        FROM read_parquet('{MPA_HEX}')
    )
""").df()

print(f"Combined area of all MPA candidates: {total_area['total_area_km2'].iloc[0]:,.0f} km²")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Combined area of all MPA candidates: 16,410,208 km²


## Data and Methods

### MPA Candidate Boundaries

Proposed MPA candidate polygons were obtained from the Global High Seas MPA Candidates dataset, indexed to H3 resolution-8 hexagonal cells (~0.74 km² each). Each candidate polygon is tiled into its constituent hex cells, with feature-level attributes (name, area, status, lead organization) replicated across rows. The `_cng_fid` column provides a unique feature identifier for deduplication.

### Fishing Effort

Annual apparent fishing effort data were sourced from Global Fishing Watch (GFW) v3 [](doi:10.1126/science.aao5646), covering 2012–2024 at H3 resolution-6 (~36 km²). Effort is derived from Automatic Identification System (AIS) vessel tracking data processed through neural network classifiers that distinguish active fishing from transit and loitering behavior.

### Spatial Analysis

All spatial operations use H3 hexagonal indexing [](doi:10.48550/arXiv.2408.13294) rather than traditional geometry predicates, enabling efficient partition-pruned joins across cloud-hosted Parquet datasets. MPA candidate hex cells (h8) are aggregated to resolution 6 via `h3_cell_to_parent` for joining with GFW data. South Atlantic candidates are identified by centroid coordinates (longitude −70° to 20°, latitude < 0°).

In [3]:
candidates = con.sql(f"""
    SELECT 
        MPA,
        ROUND(Area) AS area_km2,
        Status,
        BBNJlead AS lead,
        ROUND(AVG(h3_cell_to_lng(h8)), 2) AS avg_lon,
        ROUND(AVG(h3_cell_to_lat(h8)), 2) AS avg_lat
    FROM (
        SELECT DISTINCT h8, MPA, Area, Status, BBNJlead, _cng_fid
        FROM read_parquet('{MPA_HEX}')
    )
    GROUP BY MPA, Area, Status, BBNJlead
    ORDER BY MPA
""").df()

candidates.style.set_caption("Proposed high seas MPA candidates with area, status, and centroid coordinates.").format({"area_km2": "{:,.0f}"})

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,MPA,area_km2,Status,lead,avg_lon,avg_lat
0,Atlantic Equatorial Fracture Zone,"1,460,213",EBSA candidate,"TBD; Brasil, Senegal",-18.280000,-1.600000
1,Canary-Guinea Current Convergence Zone,"525,222",Championed proposal,Nigeria,-21.440000,7.690000
2,Central Indian Ocean Basin EBSA,"1,759,995",EBSA candidate,TBD,82.600000,-9.490000
3,Costa Rica Thermal Dome,"511,755",Championed proposal,Costa Rica,-94.770000,9.610000
4,Del Cano Rise (Prince Edward Islands),"455,927",Active scoping,"South Africa, France, Australia",45.310000,-42.610000
5,Emperor Seamount Chain,"1,382,679",Active scoping,South Korea,151.630000,37.950000
6,Lord Howe Rise,"285,685",Championed proposal,Australia,163.910000,-32.400000
7,"Pocket High Seas: Fiji, Vanuatu, Solomon Isl.","73,681",Active scoping,Pacific Nations,173.030000,-15.340000
8,Pocket High Seas: French Polynesia,"100,191",Active scoping,Pacific Nations,-157.990000,-13.720000
9,"Pocket High Seas: PNG, FSM, Fiji, Kiribati, Nauru","777,834",Active scoping,Pacific Nations,166.370000,-3.680000


## Results

### Combined Spatial Footprint

The 18 proposed high seas MPA candidates span a combined area of approximately 16.4 million km² — roughly 4.5% of the global ocean surface. The table above lists all candidates with their area, proposal status, lead organization, and approximate centroid coordinates.

### South Atlantic Candidates

Two candidates fall within the South Atlantic basin (longitude −70° to 20°, latitude < 0°): the **Atlantic Equatorial Fracture Zone** (~1.46 million km², centered at 1.6°S in the central Atlantic) and the **Walvis Ridge** (~1.41 million km², centered at 29.3°S off the coast of Namibia and Angola). Together they account for approximately 2.87 million km², or 17.5% of the total candidate area.

In [4]:
south_atlantic = candidates[
    (candidates['avg_lon'] >= -70) & 
    (candidates['avg_lon'] <= 20) & 
    (candidates['avg_lat'] < 0)
]

print(f"South Atlantic MPA candidates ({len(south_atlantic)}):")
print(f"Combined area: {south_atlantic['area_km2'].sum():,.0f} km²\n")
south_atlantic[['MPA', 'area_km2', 'Status', 'avg_lon', 'avg_lat']]

South Atlantic MPA candidates (2):
Combined area: 2,870,722 km²



,MPA,area_km2,Status,avg_lon,avg_lat
0,Atlantic Equatorial Fracture Zone,1460213.0,EBSA candidate,-18.28,-1.60
17,Walvis Ridge,1410509.0,Active scoping,1.98,-29.32


### Fishing Effort in the Walvis Ridge Candidate

The table below summarizes annual fishing and AIS presence hours within the Walvis Ridge boundary. Total fishing effort over the 13-year period was approximately 381,000 hours, with AIS presence hours roughly double the fishing hours — indicating substantial transit and loitering activity in the region.

In [5]:
GFW_HEX = 's3://public-gfw/gfw-fishing-effort/hex/h0=*/*.parquet'

fishing_effort = con.sql(f"""
    WITH walvis AS (
        SELECT DISTINCT h8, h0
        FROM read_parquet('{MPA_HEX}')
        WHERE MPA = 'Walvis Ridge'
    ),
    walvis_h6 AS (
        SELECT DISTINCT h3_cell_to_parent(h8, 6) AS h6, h0
        FROM walvis
    )
    SELECT 
        g.year,
        ROUND(SUM(g.fishing_hours)) AS fishing_hours,
        ROUND(SUM(g.ais_hours)) AS ais_hours
    FROM walvis_h6 w
    JOIN read_parquet('{GFW_HEX}') g
        ON w.h6 = g.h6 AND w.h0 = g.h0
    GROUP BY g.year
    ORDER BY g.year
""").df()

fishing_effort

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,year,fishing_hours,ais_hours
0,2012,11392.0,23221.0
1,2013,23483.0,38721.0
2,2014,27853.0,46845.0
3,2015,15365.0,40235.0
4,2016,19326.0,42752.0
5,2017,31125.0,60736.0
6,2018,37264.0,71497.0
7,2019,28311.0,59555.0
8,2020,33809.0,68826.0
9,2021,52728.0,99055.0


[](#fig-fishing-effort) shows the temporal trajectory of fishing effort. Activity roughly tripled from 11,392 hours in 2012 to a peak of 52,728 hours in 2021. The 2024 figure (24,288 hours) shows a decline, though incomplete AIS reporting for the most recent year may partially account for this.

In [6]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(fishing_effort['year'], fishing_effort['fishing_hours'], 
       color='#FF6B35', edgecolor='#D32F2F', linewidth=0.8, label='Fishing Hours')
ax.plot(fishing_effort['year'], fishing_effort['ais_hours'], 
        color='#1B4965', marker='o', linewidth=2, label='AIS Hours (all activity)')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Hours', fontsize=12)
ax.set_title('Annual Fishing Effort in Walvis Ridge MPA Candidate (2012–2024)', fontsize=14)
ax.set_xticks(fishing_effort['year'])
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

for i, row in fishing_effort.iterrows():
    ax.text(row['year'], row['fishing_hours'] + 1500, f"{row['fishing_hours']:,.0f}", 
            ha='center', va='bottom', fontsize=7, rotation=45)

plt.tight_layout()
plt.savefig("annual-fishing-walvis.png")
plt.close()
#plt.show()

print(f"\nTotal fishing hours (2012–2024): {fishing_effort['fishing_hours'].sum():,.0f}")
print(f"Peak year: {fishing_effort.loc[fishing_effort['fishing_hours'].idxmax(), 'year']} "
      f"({fishing_effort['fishing_hours'].max():,.0f} hours)")


Total fishing hours (2012–2024): 381,264
Peak year: 2021 (52,728 hours)


```{figure} annual-fishing-walvis.png
:label: fig-fishing-effort
:alt: Annual Fishing Effort in Walvis Ridge MPA Candidate (2012–2024), hours vs years.
:align: center

Annual fishing effort (orange bars) and total AIS vessel presence hours (blue line) within the Walvis Ridge MPA candidate boundary, 2012–2024. Data: Global Fishing Watch v3.
```

## Discussion

The 18 proposed high seas MPA candidates collectively cover 16.4 million km², a substantial fraction of ABNJ. The concentration of two candidates in the South Atlantic — the Atlantic Equatorial Fracture Zone and Walvis Ridge — reflects growing recognition of this basin's ecological significance, including its role as habitat for migratory pelagic species and its unique seafloor geomorphology.

The Walvis Ridge candidate shows a clear upward trend in fishing effort from 2012 through 2021, with a nearly five-fold increase over that period. The 2021 peak of 52,728 fishing hours suggests intensifying commercial interest in this region, likely driven by longline fleets targeting tuna and tuna-like species. The ratio of AIS hours to fishing hours (~2:1) indicates that vessels spend considerable time transiting through or loitering in the area, which has implications for both enforcement planning and vessel displacement modeling.

Several caveats apply. First, AIS-based effort estimates capture only vessels with active transponders; dark fishing by vessels that disable AIS is not represented. Second, the 2024 decline may partly reflect data latency rather than a genuine reduction in effort. Third, our South Atlantic delineation uses a simple bounding box rather than formal oceanographic boundaries, though results are robust to reasonable alternative definitions.

These baseline effort estimates are a critical input for the BBNJ negotiation process. Quantifying where and how much fishing occurs within candidate boundaries enables parties to assess displacement costs, design transition support mechanisms, and prioritize candidates where conservation benefits most clearly outweigh socioeconomic impacts.

+++ {"part": "data_availability"}

## Data Availability

All datasets used in this analysis are publicly available through the NRP STAC catalog at `https://s3-west.nrp-nautilus.io/public-data/stac/catalog.json`:

- **MPA Candidates**: `mpa-candidates` collection — proposed high seas MPA polygons indexed to H3 resolution 8
- **Fishing Effort**: `gfw-fishing-effort` collection — Global Fishing Watch annual fishing effort v3 (2012–2024) at H3 resolution 6, licensed under [CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/)

This notebook is fully reproducible. All queries execute against cloud-hosted Parquet files via DuckDB with no local data downloads required.

+++ {"part": "acknowledgments"}

## Acknowledgments

Fishing effort data provided by [Global Fishing Watch](https://globalfishingwatch.org) under CC BY-NC 4.0. MPA candidate boundaries curated by the Boettiger Lab. Cloud-native geospatial infrastructure provided by the [National Research Platform (NRP)](https://nrp.ai). H3 hexagonal indexing by Uber Technologies. Analysis powered by [DuckDB](https://duckdb.org), with interactive map exploration via [GeoAgent](https://geojupyter.github.io/jupyter-geoagent/).